In [0]:
# Pipeline Run Logging Notebook
# Logs run metadata after each pipeline execution

from pyspark.sql import Row
from datetime import datetime

# Get current timestamp
run_timestamp = datetime.now()

# Read row counts from each layer
bronze_count = spark.table("metadata_governance.bronze.bronze_metadata").count()
silver_count = spark.table("metadata_governance.silver.silver_metadata").count()
gold_count = spark.table("metadata_governance.gold.gold_metadata").count()

# Create log entry
log_entry = [
    Row(
        run_timestamp=run_timestamp,
        layer="bronze",
        table_name="bronze_metadata",
        row_count=bronze_count,
        status="SUCCESS"
    ),
    Row(
        run_timestamp=run_timestamp,
        layer="silver",
        table_name="silver_metadata",
        row_count=silver_count,
        status="SUCCESS"
    ),
    Row(
        run_timestamp=run_timestamp,
        layer="gold",
        table_name="gold_metadata",
        row_count=gold_count,
        status="SUCCESS"
    )
]

# Write to monitoring Delta table
log_df = spark.createDataFrame(log_entry)
log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("metadata_governance.default.pipeline_run_log")

print(f"Pipeline run logged at {run_timestamp}")
print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Gold rows: {gold_count}")